# Phase 1: DATA


## 1.1: Download and validation

In [1]:
# import section and parameters
from __future__ import annotations

import logging
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
logger = logging.getLogger("ko_pep")

TICKERS: tuple[str, str] = ("KO", "PEP")
DATA_START = date(1990, 1, 1)
AS_OF = date(2026, 10, 9)          # last close used in the study (Friday); fixed for reproducibility
IS_END = date(2014, 12, 31)        # pre-registered split
OOS_START = date(2015, 1, 1)

RAW_PATH = Path("data/raw/prices_raw.pkl")
VALIDATED_PATH = Path("data/processed/prices_validated.pkl")

class DataFetchError(RuntimeError):
    """Raised when the data source returns no usable data."""


class StaleDataError(RuntimeError):
    """Raised when the last available close is older than expected."""

In [2]:
def fetch_prices(
    tickers: tuple[str, ...],
    start: date,
    as_of: date,
    path: Path,
    refresh: bool = False,
) -> pd.DataFrame:
    """Download dividend-adjusted daily prices, or load them from the raw checkpoint."""

    if path.exists() and not refresh:
        logger.info("Loading raw checkpoint %s", path)
        return pd.read_pickle(path)

    logger.info("Downloading %s from %s to %s", tickers, start, as_of)
    raw = yf.download(
        list(tickers),
        start=start.isoformat(),
        end=(as_of + timedelta(days=1)).isoformat(),
        auto_adjust=True,
        progress=False,
    )
    if raw is None or raw.empty:
        raise DataFetchError(f"No data returned for {tickers}")

    path.parent.mkdir(parents=True, exist_ok=True)
    raw.to_pickle(path)
    logger.info("Saved raw checkpoint %s (%d rows)", path, len(raw))
    return raw

In [3]:
def validate_prices(
    raw: pd.DataFrame,
    tickers: tuple[str, ...],
    as_of: date,
    max_staleness_days: int = 5,
    jump_threshold: float = 0.20,
) -> pd.DataFrame:

    """
    Parameters
    ----------
    raw : pd.DataFrame
        Output of ``fetch_prices``.
    tickers : tuple[str, ...]
        Symbols to keep, in order.
    as_of : date
        No observation after this date is allowed.
    max_staleness_days : int, default 5
        Maximum calendar-day gap between ``as_of`` and the last close.
    jump_threshold : float, default 0.20
        Absolute daily log return above which a warning is logged.

    Returns
    -------
    pd.DataFrame
        Close prices indexed by date, one column per ticker, no missing values.

    Raises
    ------
    StaleDataError
        If the last close is older than ``max_staleness_days``."""


    close = raw["Close"].loc[:, list(tickers)].copy()
    close.index = pd.to_datetime(close.index)
    close = close.loc[close.index <= pd.Timestamp(as_of)]

    n_missing = close.isna().sum()
    if n_missing.any():
        logger.warning("Dropping rows with missing prices: %s", n_missing.to_dict())
    close = close.dropna(how="any")

    assert close.index.is_monotonic_increasing and close.index.is_unique, "Index must be sorted and unique"
    assert (close > 0).all().all(), "Prices must be strictly positive"
    assert close.index.max() <= pd.Timestamp(as_of), "No data after as_of"

    last_close = close.index.max().date()
    if (as_of - last_close).days > max_staleness_days:
        raise StaleDataError(f"Last close {last_close} is older than {max_staleness_days} days before {as_of}")

    log_ret = np.log(close).diff().dropna()
    jumps = log_ret[(log_ret.abs() > jump_threshold).any(axis=1)]
    if not jumps.empty:
        logger.warning("Large daily moves to inspect:\n%s", jumps)

    return close

In [4]:
raw = fetch_prices(TICKERS, DATA_START, AS_OF, RAW_PATH)
prices = validate_prices(raw, TICKERS, AS_OF)

raw_flat = raw.copy()
raw_flat.columns = [f"{ticker}_{field}" for field, ticker in raw_flat.columns]
raw_flat.to_csv(RAW_PATH.with_suffix(".csv"), index_label="date")

VALIDATED_PATH.parent.mkdir(parents=True, exist_ok=True)
prices.to_pickle(VALIDATED_PATH)
prices.to_csv(VALIDATED_PATH.with_suffix(".csv"), index_label="date", float_format="%.6f")

is_mask = prices.index <= pd.Timestamp(IS_END)
prices_is = prices.loc[: pd.Timestamp(IS_END)]
assert prices_is.index.max() <= pd.Timestamp(IS_END), "In-sample frame must not contain OOS dates"
oos_mask = prices.index >= pd.Timestamp(OOS_START)
assert is_mask.sum() + oos_mask.sum() == len(prices), "IS and OOS must cover all rows exactly once"

summary = pd.DataFrame(
    {
        "first date": [prices.index.min().date()],
        "last date": [prices.index.max().date()],
        "observations": [len(prices)],
        "in-sample obs": [int(is_mask.sum())],
        "out-of-sample obs": [int(oos_mask.sum())],
    }
)
summary

INFO | Loading raw checkpoint data\raw\prices_raw.pkl


,first date,last date,observations,in-sample obs,out-of-sample obs
0,1990-01-02,2026-10-09,9261,6301,2960


## 1.2: Prices and price ratio

In [5]:
normalised = prices_is / prices_is.iloc[0] * 100

fig = go.Figure()
for ticker in TICKERS:
    fig.add_trace(go.Scatter(x=normalised.index, y=normalised[ticker], name=ticker, mode="lines"))
fig.update_layout(
    title="KO and PEP, dividend-adjusted, rebased to 100 (log scale) in-sample 1990-2014",
    yaxis_type="log",
    yaxis_title="Index (start = 100)",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

In [6]:
ratio = prices_is["KO"] / prices_is["PEP"]

fig = go.Figure(go.Scatter(x=ratio.index, y=ratio, mode="lines", name="KO / PEP"))
fig.update_layout(
    title="Price ratio KO / PEP in-sample 1990-2014",
    yaxis_title="Ratio",
    template="plotly_white",
)
fig.show()

Three regimes are observed:
- 1990-1998 the ratio grows from 0.5 to 1 approximately, with a peak in August 1998
- 1998-2005 the ratio decreases to 0.35/0.40 cancelling the previous growth
- 2005-2014 the ratio steadily oscillates between 0.35 and 0.55

## 1.3: Return correlation

In [7]:
log_ret = np.log(prices_is).diff().dropna()

periods = {"In-sample (1990-2014)": log_ret}
for decade, chunk in log_ret.groupby((log_ret.index.year // 10) * 10):
    periods[f"{decade}s"] = chunk

corr_table = pd.DataFrame(
    {
        "correlation": [chunk["KO"].corr(chunk["PEP"]) for chunk in periods.values()],
        "observations": [len(chunk) for chunk in periods.values()],
    },
    index=list(periods.keys()),
).round(3)
assert corr_table["correlation"].between(-1, 1).all()
assert log_ret.index.max() <= pd.Timestamp(IS_END), "Correlation must use in-sample data only"

rolling_corr = log_ret["KO"].rolling(252).corr(log_ret["PEP"])

fig = go.Figure(go.Scatter(x=rolling_corr.index, y=rolling_corr, mode="lines", name="1-year rolling correlation"))
fig.update_layout(
    title="1-year rolling correlation of daily log returns, KO vs PEP in-sample 1990-2014",
    yaxis_title="Correlation",
    yaxis_range=[-0.2, 1],
    template="plotly_white",
)
fig.show()
corr_table

,correlation,observations
In-sample (1990-2014),0.514,6300
1990s,0.508,2527
2000s,0.512,2515
2010s,0.602,1258


The return correlation is always positive but clearly time-varying. The full in-sample value is 0.51, but the 1-year rolling estimate ranges from about
0.2 to 0.8. 

The 2008-09 peak (~0.75) is probably a market-wide effect, since correlations rise across most stocks in a crisis. 

From 2008 to 2014 the correlation stays higher (0.55-0.7), in line with the more stable ratio.


# Phase 2: Build mean-reversion signal on the spread and validate it

## 2.1: Build the spread

The hedge ratio $\beta$ says how many PEP shares to trade against one KO share so that the position
does not depend on the common movement of the two stocks. We estimate it once by OLS on in-sample
prices and then freeze it: the out-of-sample period never influences it.

In [8]:
from dataclasses import dataclass

import statsmodels.api as sm


@dataclass(frozen=True)
class HedgeFit:
    """Result of the static OLS hedge-ratio regression.

    Attributes
    ----------
    alpha : float
        Intercept of the regression.
    beta : float
        Hedge ratio: units of ``x`` per unit of ``y``.
    n_obs : int
        Number of observations used.
    fit_start : date
        First date used in the fit.
    fit_end : date
        Last date used in the fit.
    """

    alpha: float
    beta: float
    n_obs: int
    fit_start: date
    fit_end: date


def fit_hedge_ratio(prices: pd.DataFrame, y: str, x: str, as_of: date) -> HedgeFit:
    """Estimate a static hedge ratio by OLS of ``y`` on ``x``, using data up to ``as_of``.

    Parameters
    ----------
    prices : pd.DataFrame
        Price levels indexed by date, one column per ticker.
    y : str
        Dependent stock (bought when the spread is long).
    x : str
        Independent stock (sold, in ``beta`` units, when the spread is long).
    as_of : date
        Last date allowed in the fit (inclusive).

    Returns
    -------
    HedgeFit
        Estimated coefficients and fit metadata.

    Notes
    -----
    The regression is run on price levels, as the brief prescribes. Because both
    series are non-stationary, the usual OLS t-statistics and R² are not
    meaningful here (spurious-regression problem); only the point estimate of
    beta is used. Whether the resulting spread is stationary is tested separately
    in step 2.
    """
    data = prices.loc[prices.index <= pd.Timestamp(as_of), [y, x]]
    assert data.index.max() <= pd.Timestamp(as_of), "Fit must not use data after as_of"
    assert not data.isna().any().any(), "Fit data must have no missing values"

    model = sm.OLS(data[y], sm.add_constant(data[x])).fit()
    beta = float(model.params[x])
    assert beta > 0, "A negative hedge ratio would not make economic sense for this pair"

    return HedgeFit(
        alpha=float(model.params["const"]),
        beta=beta,
        n_obs=int(model.nobs),
        fit_start=data.index.min().date(),
        fit_end=data.index.max().date(),
    )


def compute_spread(prices: pd.DataFrame, fit: HedgeFit, y: str, x: str) -> pd.Series:
    """Compute the spread ``y - beta * x`` with a frozen hedge ratio.

    Parameters
    ----------
    prices : pd.DataFrame
        Price levels indexed by date.
    fit : HedgeFit
        Hedge ratio estimated beforehand; it is not re-estimated here.
    y : str
        Dependent stock.
    x : str
        Independent stock.

    Returns
    -------
    pd.Series
        Spread in price units (dollars), indexed by date.

    Notes
    -----
    The intercept is not subtracted, following the brief's definition. The spread
    therefore oscillates around ``alpha`` rather than around zero; the z-score in
    step 3 removes the level anyway.
    """
    return (prices[y] - fit.beta * prices[x]).rename("spread")

In [9]:
hedge = fit_hedge_ratio(prices_is, y="KO", x="PEP", as_of=IS_END)
spread_is = compute_spread(prices_is, hedge, y="KO", x="PEP")

assert hedge.fit_end <= IS_END, "Hedge ratio must be estimated on in-sample data only"
assert hedge.n_obs == len(prices_is), "Fit must use every in-sample observation"
assert abs(spread_is.mean() - hedge.alpha) < 1e-8, "With an intercept, the mean spread must equal alpha"

logger.info("Hedge ratio fitted on %s → %s: beta = %.4f", hedge.fit_start, hedge.fit_end, hedge.beta)

pd.DataFrame(
    {
        "alpha": [round(hedge.alpha, 4)],
        "beta (hedge ratio)": [round(hedge.beta, 4)],
        "observations": [hedge.n_obs],
        "fit period": [f"{hedge.fit_start} → {hedge.fit_end}"],
    }
)

INFO | Hedge ratio fitted on 1990-01-02 → 2014-12-31: beta = 0.3988


,alpha,beta (hedge ratio),observations,fit period
0,2.8729,0.3988,6301,1990-01-02 → 2014-12-31


In [10]:
spread_mean = spread_is.mean()
spread_std = spread_is.std()
logger.info(
    "In-sample spread: mean = %.4f USD, std = %.4f USD, ±2std band = [%.4f, %.4f]",
    spread_mean,
    spread_std,
    spread_mean - 2 * spread_std,
    spread_mean + 2 * spread_std,
)

fig = go.Figure(go.Scatter(x=spread_is.index, y=spread_is, mode="lines", name="Spread"))
fig.add_hline(y=spread_mean, line_color="black", line_width=1)
for k in (-2, 2):
    fig.add_hline(y=spread_mean + k * spread_std, line_dash="dash", line_color="grey", line_width=1)
fig.update_layout(
    title=f"Spread KO - {hedge.beta:.3f}·PEP in-sample 1990-2014 (solid: mean, dashed: ±2std)",
    yaxis_title="Spread (USD)",
    template="plotly_white",
)
fig.show()

INFO | In-sample spread: mean = 2.8729 USD, std = 2.9030 USD, ±2std band = [-2.9330, 8.6788]


The spread does not look like a fast mean-reverting series. Its excursions last years:
below the mean in 1990-1995, above it in 1995-2001 (peaking near 13 USD in 1998 and staying above +2std in 1998), below it in 2001-2010 (low near −2.2 in 2006), and above again in 2011-2014.

It stays on one side of its mean for years at a time, and never reaches −2std. If there is a mean reversion, it is very slow.

Next I check the mean reversion quantitatively counting the spread's mean crossings in-sample and compare the count with 1,000 simulated random walks of the same length, which have no mean reversion by construction. If most of the random walks cross their mean more often than the spread, that could be evidence
that there is no mean reversion.

I then estimate the half-life of the possible mean reversion from an AR(1) fit on the spread: the average number of trading days needed to close half of a deviation from the mean. This tells us on what time scale the mean reversion can operate.


In [11]:
def count_mean_crossings(values: pd.Series | np.ndarray) -> int:
    """Count how many times a series crosses its own sample mean.

    Parameters
    ----------
    values : pd.Series or np.ndarray
        The series to analyse.

    Returns
    -------
    int
        Number of sign changes of ``values - mean(values)``.

    Notes
    -----
    This is the check suggested in the brief ("does it keep crossing back over its
    own historical average"). Observations exactly equal to the mean are ignored.
    """
    x = np.asarray(values, dtype=float)
    signs = np.sign(x - x.mean())
    signs = signs[signs != 0]
    return int(np.count_nonzero(np.diff(signs)))


def random_walk_crossings(n_steps: int, n_paths: int, seed: int) -> np.ndarray:
    """Simulate random walks and count how often each crosses its own mean.

    Parameters
    ----------
    n_steps : int
        Length of each simulated path (same as the real series).
    n_paths : int
        Number of simulated paths.
    seed : int
        Random seed, for reproducibility.

    Returns
    -------
    np.ndarray
        Number of mean crossings for each path.

    Notes
    -----
    Benchmark for ``count_mean_crossings``: how many crossings a series with no
    mean reversion at all produces by chance. The step size does not matter,
    because the crossing count is unchanged by rescaling the path.
    """
    rng = np.random.default_rng(seed)
    crossings = np.empty(n_paths, dtype=int)
    for i in range(n_paths):
        path = np.cumsum(rng.standard_normal(n_steps))
        crossings[i] = count_mean_crossings(path)
    return crossings


def half_life(series: pd.Series) -> float:
    """Estimate the mean-reversion half-life of a series from an AR(1) fit.

    Parameters
    ----------
    series : pd.Series
        The series to analyse, indexed by date.

    Returns
    -------
    float
        Half-life in observations (trading days); ``inf`` if there is no pull
        back toward the mean.

    Notes
    -----
    Fits ``Δs_t = a + b·s_{t-1} + e_t``, which is an AR(1) written in differences.
    A negative ``b`` means the series is pulled back toward its mean; the
    half-life is ``-ln 2 / ln(1 + b)``, the number of days needed to close half
    of a deviation.
    """
    lagged = series.shift(1).iloc[1:]
    delta = series.diff().iloc[1:]
    model = sm.OLS(delta, sm.add_constant(lagged)).fit()
    b = float(model.params.iloc[1])
    if b >= 0:
        return float("inf")
    return float(-np.log(2) / np.log(1 + b))

In [12]:
n_crossings = count_mean_crossings(spread_is)
rw_crossings = random_walk_crossings(n_steps=len(spread_is), n_paths=1000, seed=42)
share_rw_at_least = float((rw_crossings >= n_crossings).mean())
hl_days = half_life(spread_is)

assert n_crossings >= 0, "Crossing count cannot be negative"
assert 0.0 <= share_rw_at_least <= 1.0, "Share must be a probability"
assert spread_is.index.max() <= pd.Timestamp(IS_END), "Diagnostics must use in-sample data only"

logger.info(
    "Spread crosses its mean %d times; %.1f%% of random walks cross at least as often; half-life %.0f days",
    n_crossings,
    100 * share_rw_at_least,
    hl_days,
)

stationarity = pd.DataFrame(
    {
        "value": [
            n_crossings,
            int(np.median(rw_crossings)),
            f"{100 * share_rw_at_least:.1f}%",
            f"{hl_days:.0f} days" if np.isfinite(hl_days) else "no reversion",
        ]
    },
    index=[
        "Mean crossings of the spread (1990-2014)",
        "Median crossings of a random walk (1,000 paths)",
        "Share of random walks crossing at least as often",
        "Half-life of mean reversion (AR(1))",
    ],
)
stationarity

INFO | Spread crosses its mean 47 times; 61.6% of random walks cross at least as often; half-life 387 days


,value
Mean crossings of the spread (1990-2014),47
"Median crossings of a random walk (1,000 paths)",59
Share of random walks crossing at least as often,61.6%
Half-life of mean reversion (AR(1)),387 days


The spread crosses its mean 47 times in 1990-2014, fewer than the median random walk of
the same length (59); 61.6% of 1,000 simulated random walks cross at least as often. In-sample, the
spread is indistinguishable from a random walk.

The AR(1) fit gives a half-life of 387 trading days (about 1.5 years). If there is any
long-run mean reversion, it is very slow. Since an AR(1) fit on a finite random walk also yields a
finite half-life, this number is not evidence of mean reversion in itself, it indicates the time
scale on which any reversion would operate.


There is no in-sample evidence of long-run mean reversion. The
60-day z-score window used next is about six times shorter than the half-life, so the backtest does
not test a return to the long-run mean: it tests whether short-term deviations from a moving local
mean close within weeks.

## 2.2: Signal and in-sample backtest

The signal is a rolling z-score of the spread:

$$z_t = \frac{s_t - \mu_t}{\sigma_t}$$

where $\mu_t$ and $\sigma_t$ are the mean and standard deviation over the last $L$ days.
When $z_t > $ entry we short the spread (sell KO, buy $\beta$ PEP); when $z_t < -$entry we buy it.
We close when $z$ crosses the exit level (0). Positions are applied from the next day, so a trade never
uses information it could not have had. Returns are divided by gross exposure, and each change in
position pays the transaction cost on both stocks.

In [13]:
import sqlite3
from datetime import datetime

from plotly.subplots import make_subplots


def rolling_zscore(spread: pd.Series, lookback: int) -> pd.Series:
    """Standardise the spread against its own rolling mean and standard deviation.

    Parameters
    ----------
    spread : pd.Series
        Spread indexed by date.
    lookback : int
        Window length in trading days.

    Returns
    -------
    pd.Series
        z-score; NaN for the first ``lookback - 1`` days.

    Notes
    -----
    The window ends at (and includes) day t, so z_t uses only data observable at
    the close of t. Positions derived from z_t must therefore earn returns from
    t+1 onward, which ``backtest`` enforces with a one-day shift.
    """
    mean = spread.rolling(lookback, min_periods=lookback).mean()
    std = spread.rolling(lookback, min_periods=lookback).std()
    return ((spread - mean) / std).rename("zscore")


def generate_positions(z: pd.Series, entry_z: float, exit_z: float) -> pd.Series:
    """Turn a z-score into spread positions with entry and exit thresholds.

    Parameters
    ----------
    z : pd.Series
        z-score of the spread.
    entry_z : float
        Open a position when abs(z) exceeds this threshold.
    exit_z : float
        Close a long when z >= -exit_z, close a short when z <= exit_z.

    Returns
    -------
    pd.Series
        +1 (long spread: long y, short beta·x), -1 (short spread) or 0, decided
        at the close of each day.

    Raises
    ------
    ValueError
        If ``entry_z <= exit_z`` or ``exit_z < 0``.

    Notes
    -----
    A position can be closed and the opposite one opened on the same day if z
    jumps across both thresholds. Days with an undefined z-score (warm-up) are flat.
    """
    if not entry_z > exit_z >= 0:
        raise ValueError("Require entry_z > exit_z >= 0")

    values = z.to_numpy()
    positions = np.zeros(len(values))
    current = 0
    for i, zi in enumerate(values):
        if np.isnan(zi):
            current = 0
        else:
            if current == 1 and zi >= -exit_z:
                current = 0
            elif current == -1 and zi <= exit_z:
                current = 0
            if current == 0:
                if zi > entry_z:
                    current = -1
                elif zi < -entry_z:
                    current = 1
        positions[i] = current
    return pd.Series(positions, index=z.index, name="position")


def backtest(
    prices: pd.DataFrame,
    fit: HedgeFit,
    y: str,
    x: str,
    positions: pd.Series,
    cost_bps: float,
) -> pd.DataFrame:
    """Compute daily returns of a spread strategy, net of transaction costs.

    Parameters
    ----------
    prices : pd.DataFrame
        Price levels indexed by date.
    fit : HedgeFit
        Frozen hedge ratio.
    y : str
        Dependent stock.
    x : str
        Independent stock.
    positions : pd.Series
        Positions decided at each close (output of ``generate_positions``).
    cost_bps : float
        Cost per stock per trade, in basis points of traded notional.

    Returns
    -------
    pd.DataFrame
        Columns: position, held, gross_return, cost, net_return, equity.

    Notes
    -----
    ``held`` is the position decided the previous day; it alone earns day t's
    return, which rules out look-ahead. Returns are spread P&L divided by the
    previous day's gross exposure (y + beta·x). Trading one unit of the spread
    trades a notional equal to the gross exposure across both stocks, so the cost
    in return terms is ``abs(change in position) * cost_bps / 1e4``.
    """
    assert positions.index.equals(prices.index), "Positions and prices must share the same dates"

    spread = compute_spread(prices, fit, y, x)
    gross_exposure = prices[y] + fit.beta * prices[x]
    held = positions.shift(1).fillna(0.0)

    gross_return = (held * spread.diff() / gross_exposure.shift(1)).fillna(0.0)
    turnover = positions.diff().abs().fillna(positions.abs())
    cost = turnover * cost_bps / 1e4
    net_return = gross_return - cost

    result = pd.DataFrame(
        {
            "position": positions,
            "held": held,
            "gross_return": gross_return,
            "cost": cost,
            "net_return": net_return,
            "equity": (1 + net_return).cumprod(),
        }
    )
    assert np.allclose(result["gross_return"] - result["cost"], result["net_return"])
    assert (result.loc[result["held"] == 0, "gross_return"] == 0).all(), "Flat days must earn nothing"
    return result


def performance_summary(bt: pd.DataFrame, cost_bps: float, periods_per_year: int = 252) -> dict[str, float]:
    """Summarise a backtest with annualised and per-trade statistics.

    Parameters
    ----------
    bt : pd.DataFrame
        Output of ``backtest``.
    cost_bps : float
        Cost level used, to charge entry and exit costs to each trade.
    periods_per_year : int, default 252
        Annualisation factor.

    Returns
    -------
    dict[str, float]
        sharpe, ann_return, ann_vol, max_drawdown, n_trades, hit_rate,
        avg_holding_days, time_in_market.

    Notes
    -----
    Sharpe and volatility include flat days (zero return), so they describe the
    strategy as a whole, not only the days in the market. Per-trade returns are
    the sum of daily gross returns while the trade is held, minus an entry and an
    exit cost; this is an approximation, adequate for hit rate and counts.
    """
    r = bt["net_return"]
    vol = r.std() * np.sqrt(periods_per_year)
    sharpe = r.mean() * periods_per_year / vol if vol > 0 else float("nan")
    drawdown = bt["equity"] / bt["equity"].cummax() - 1

    pos = bt["position"]
    starts = (pos != 0) & (pos != pos.shift(1).fillna(0))
    trade_id = starts.cumsum().where(pos != 0)
    held_trade_id = trade_id.shift(1).where(bt["held"] != 0)

    per_trade = bt["gross_return"].groupby(held_trade_id).sum() - 2 * cost_bps / 1e4
    holding = bt["held"].groupby(held_trade_id).size()

    return {
        "sharpe": float(sharpe),
        "ann_return": float(r.mean() * periods_per_year),
        "ann_vol": float(vol),
        "max_drawdown": float(drawdown.min()),
        "n_trades": int(starts.sum()),
        "hit_rate": float((per_trade > 0).mean()) if len(per_trade) else float("nan"),
        "avg_holding_days": float(holding.mean()) if len(holding) else float("nan"),
        "time_in_market": float((bt["held"] != 0).mean()),
    }

In [14]:
TRIALS_DB = Path("results/trials.sqlite")


def open_trial_registry(path: Path) -> sqlite3.Connection:
    """Open (or create) the SQLite registry of backtest trials.

    Parameters
    ----------
    path : Path
        Location of the SQLite file.

    Returns
    -------
    sqlite3.Connection
        Open connection with the ``trials`` table in place.

    Notes
    -----
    One row per distinct configuration (period, lookback, entry, exit, cost).
    Re-running a configuration updates its metrics and ``last_run_at`` but keeps
    ``first_run_at`` and the original label, so the row count is the number of
    configurations ever tried.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(path)
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS trials (
            period TEXT, lookback INTEGER, entry_z REAL, exit_z REAL, cost_bps REAL,
            label TEXT, beta REAL,
            sharpe REAL, ann_return REAL, max_drawdown REAL, n_trades INTEGER, hit_rate REAL,
            first_run_at TEXT, last_run_at TEXT,
            PRIMARY KEY (period, lookback, entry_z, exit_z, cost_bps)
        )
        """
    )
    return conn


def log_trial(
    conn: sqlite3.Connection,
    period: str,
    lookback: int,
    entry_z: float,
    exit_z: float,
    cost_bps: float,
    label: str,
    beta: float,
    metrics: dict[str, float],
) -> None:
    """Insert a trial, or update its metrics if the configuration was already run.

    Parameters
    ----------
    conn : sqlite3.Connection
        Open registry.
    period : str
        "in-sample" or "out-of-sample".
    lookback, entry_z, exit_z, cost_bps
        Configuration of the rule.
    label : str
        Why the trial was run (e.g. "baseline", "grid", "cost sensitivity").
    beta : float
        Hedge ratio used.
    metrics : dict[str, float]
        Output of ``performance_summary``.
    """
    now = datetime.now().isoformat(timespec="seconds")
    conn.execute(
        """
        INSERT INTO trials VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT (period, lookback, entry_z, exit_z, cost_bps) DO UPDATE SET
            sharpe = excluded.sharpe, ann_return = excluded.ann_return,
            max_drawdown = excluded.max_drawdown, n_trades = excluded.n_trades,
            hit_rate = excluded.hit_rate, beta = excluded.beta, last_run_at = excluded.last_run_at
        """,
        (
            period, lookback, entry_z, exit_z, cost_bps, label, beta,
            metrics["sharpe"], metrics["ann_return"], metrics["max_drawdown"],
            metrics["n_trades"], metrics["hit_rate"], now, now,
        ),
    )
    conn.commit()
    logger.info(
        "Trial logged: %s, lookback=%d, entry=%.1f, exit=%.1f, cost=%.0f bp → Sharpe %.2f",
        period, lookback, entry_z, exit_z, cost_bps, metrics["sharpe"],
    )

### 2.2.1: Baseline rule

Let's fix a baseline to try the strategy on in-the-sample data as a 60-day rolling z-score (about one quarter of trading days), entry when |z| > 2 (the standard pairs-trading threshold), exit when z returns to 0.

I already included the transactions costs from the optional extensions as each position change
pays 5 bp of the traded notional on each stock; a direct flip from long to short pays twice. I also
report 0 bp and 10 bp to show how much the result depends on this assumption

In [15]:
BASE_LOOKBACK = 60
BASE_ENTRY = 2.0
BASE_EXIT = 0.0
BASE_COST = 5.0
COST_LEVELS = (0.0, 5.0, 10.0)

registry = open_trial_registry(TRIALS_DB)

z_is = rolling_zscore(spread_is, BASE_LOOKBACK)
pos_is = generate_positions(z_is, BASE_ENTRY, BASE_EXIT)

assert set(pos_is.unique()) <= {-1.0, 0.0, 1.0}, "Positions must be -1, 0 or +1"
assert (pos_is.iloc[: BASE_LOOKBACK - 1] == 0).all(), "No position before the z-score is defined"
truncated = rolling_zscore(spread_is.iloc[:3000], BASE_LOOKBACK)
assert np.allclose(truncated.dropna(), z_is.iloc[:3000].dropna()), "z-score must not depend on future data"

baseline_results = {}
for cost in COST_LEVELS:
    bt = backtest(prices_is, hedge, "KO", "PEP", pos_is, cost)
    metrics = performance_summary(bt, cost)
    label = "baseline" if cost == BASE_COST else "cost sensitivity"
    log_trial(registry, "in-sample", BASE_LOOKBACK, BASE_ENTRY, BASE_EXIT, cost, label, hedge.beta, metrics)
    baseline_results[f"{cost:g} bp"] = metrics

bt_is = backtest(prices_is, hedge, "KO", "PEP", pos_is, BASE_COST)

pd.DataFrame(baseline_results).T.style.format(
    {
        "sharpe": "{:.2f}",
        "ann_return": "{:.1%}",
        "ann_vol": "{:.1%}",
        "max_drawdown": "{:.1%}",
        "n_trades": "{:.0f}",
        "hit_rate": "{:.0%}",
        "avg_holding_days": "{:.0f}",
        "time_in_market": "{:.0%}",
    }
)

INFO | Trial logged: in-sample, lookback=60, entry=2.0, exit=0.0, cost=0 bp → Sharpe 0.50
INFO | Trial logged: in-sample, lookback=60, entry=2.0, exit=0.0, cost=5 bp → Sharpe 0.45
INFO | Trial logged: in-sample, lookback=60, entry=2.0, exit=0.0, cost=10 bp → Sharpe 0.41


,sharpe,ann_return,ann_vol,max_drawdown,n_trades,hit_rate,avg_holding_days,time_in_market
0 bp,0.50,4.6%,9.2%,-21.5%,106,75%,33,55%
5 bp,0.45,4.2%,9.2%,-21.5%,106,75%,33,55%
10 bp,0.41,3.7%,9.2%,-21.5%,106,75%,33,55%


In [16]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=(
        f"z-score ({BASE_LOOKBACK}-day window) with entry thresholds ±{BASE_ENTRY:g}",
        "Equity curve, in-sample (start = 1)",
    ),
)
fig.add_trace(go.Scatter(x=z_is.index, y=z_is, mode="lines", name="z-score", line_width=1), row=1, col=1)
for level in (-BASE_ENTRY, 0.0, BASE_ENTRY):
    fig.add_hline(y=level, line_dash="dash" if level else "solid", line_color="grey", line_width=1, row=1, col=1)

for cost in (0.0, BASE_COST):
    equity = backtest(prices_is, hedge, "KO", "PEP", pos_is, cost)["equity"]
    fig.add_trace(go.Scatter(x=equity.index, y=equity, mode="lines", name=f"Equity, {cost:g} bp"), row=2, col=1)

fig.update_layout(
    title="Baseline rule, in-sample 1990-2014",
    height=650,
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

The in-sample equity curve shows no gain from 1990 to 1998. From 1999 to 2004 the equity rises from about 1 to 2.2 (net of costs). From 2004 to 2014  the equity moves sideways standing in between about 2.2 and 2.3 with a drawdown in 2010-11 and a final jump at the end of 2014.
The costs open a gap between the 0 bp and 5 bp curves that widens slowly over time, but they do not change the shape as the strategy trades few times a year.

### 2.2.2: Sensitivity grid

To check the sensitivity,  I also run a planned grid: entry {1.5, 2.0, 2.5} × lookback {20, 60, 120} = 9 configurations

In [17]:
GRID_LOOKBACK = (20, 60, 120)
GRID_ENTRY = (1.5, 2.0, 2.5)

grid_rows = []
for lookback in GRID_LOOKBACK:
    z = rolling_zscore(spread_is, lookback)
    for entry in GRID_ENTRY:
        positions = generate_positions(z, entry, BASE_EXIT)
        bt = backtest(prices_is, hedge, "KO", "PEP", positions, BASE_COST)
        metrics = performance_summary(bt, BASE_COST)
        log_trial(registry, "in-sample", lookback, entry, BASE_EXIT, BASE_COST, "grid", hedge.beta, metrics)
        grid_rows.append({"lookback": lookback, "entry_z": entry, **metrics})

grid = pd.DataFrame(grid_rows)
assert len(grid) == len(GRID_LOOKBACK) * len(GRID_ENTRY) == 9

sharpe_matrix = grid.pivot(index="lookback", columns="entry_z", values="sharpe")
fig = go.Figure(
    go.Heatmap(
        z=sharpe_matrix.to_numpy(),
        x=[f"entry {e:g} std" for e in sharpe_matrix.columns],
        y=[f"{lb} days" for lb in sharpe_matrix.index],
        colorscale="RdBu",
        zmid=0,
        text=sharpe_matrix.round(2).to_numpy(),
        texttemplate="%{text}",
        colorbar_title="Sharpe",
    )
)
fig.update_layout(
    title=f"In-sample Sharpe, net of {BASE_COST:g} bp baseline is 60 days / 2 std",
    yaxis_title="z-score lookback",
    template="plotly_white",
)
fig.show()

n_rule_trials = registry.execute(
    "SELECT COUNT(*) FROM trials WHERE period = 'in-sample' AND cost_bps = ?", (BASE_COST,)
).fetchone()[0]
logger.info("Distinct in-sample rule configurations tried so far: %d", n_rule_trials)

grid.set_index(["lookback", "entry_z"]).round(3)

INFO | Trial logged: in-sample, lookback=20, entry=1.5, exit=0.0, cost=5 bp → Sharpe 0.20
INFO | Trial logged: in-sample, lookback=20, entry=2.0, exit=0.0, cost=5 bp → Sharpe 0.34
INFO | Trial logged: in-sample, lookback=20, entry=2.5, exit=0.0, cost=5 bp → Sharpe 0.05
INFO | Trial logged: in-sample, lookback=60, entry=1.5, exit=0.0, cost=5 bp → Sharpe 0.34
INFO | Trial logged: in-sample, lookback=60, entry=2.0, exit=0.0, cost=5 bp → Sharpe 0.45
INFO | Trial logged: in-sample, lookback=60, entry=2.5, exit=0.0, cost=5 bp → Sharpe 0.20
INFO | Trial logged: in-sample, lookback=120, entry=1.5, exit=0.0, cost=5 bp → Sharpe 0.38
INFO | Trial logged: in-sample, lookback=120, entry=2.0, exit=0.0, cost=5 bp → Sharpe 0.50
INFO | Trial logged: in-sample, lookback=120, entry=2.5, exit=0.0, cost=5 bp → Sharpe 0.40


INFO | Distinct in-sample rule configurations tried so far: 12


sharpe  ann_return  ann_vol  max_drawdown  n_trades  \
lookback entry_z                                                        
20       1.5       0.197       0.020    0.101        -0.255       329   
         2.0       0.341       0.031    0.090        -0.272       238   
         2.5       0.049       0.003    0.067        -0.242       125   
60       1.5       0.342       0.035    0.102        -0.253       139   
         2.0       0.453       0.042    0.092        -0.215       106   
         2.5       0.197       0.014    0.072        -0.387        61   
120      1.5       0.379       0.039    0.102        -0.342        82   
         2.0       0.504       0.046    0.091        -0.265        66   
         2.5       0.398       0.028    0.071        -0.289        42   

                  hit_rate  avg_holding_days  time_in_market  
lookback entry_z                                              
20       1.5         0.657            12.894           0.673  
         2.0         0.655            13.340           0.504  
         2.5         0.640            15.944           0.316  
60       1.5         0.763            30.597           0.675  
         2.0         0.755            32.811           0.552  
         2.5         0.689            39.852           0.386  
120      1.5         0.768            53.159           0.692  
         2.0         0.803            55.864           0.585  
         2.5         0.833            63.786           0.425

Since the Sharpe seems to increase with the length of the lookback window, I tried one more test with a lookback of one entire trading year.

In [18]:
EXTRA_LOOKBACK = 252  # added after seeing the grid: performance rose with the lookback, and the half-life is ~387 days

z_252 = rolling_zscore(spread_is, EXTRA_LOOKBACK)
extra_rows = []
for entry in GRID_ENTRY:
    positions = generate_positions(z_252, entry, BASE_EXIT)
    bt = backtest(prices_is, hedge, "KO", "PEP", positions, BASE_COST)
    metrics = performance_summary(bt, BASE_COST)
    log_trial(registry, "in-sample", EXTRA_LOOKBACK, entry, BASE_EXIT, BASE_COST,
              "grid extension (post hoc)", hedge.beta, metrics)
    extra_rows.append({"lookback": EXTRA_LOOKBACK, "entry_z": entry, **metrics})

extra_results = pd.DataFrame(extra_rows).set_index(["lookback", "entry_z"])

n_rule_trials = registry.execute(
    "SELECT COUNT(*) FROM trials WHERE period = 'in-sample' AND cost_bps = ?", (BASE_COST,)
).fetchone()[0]
assert n_rule_trials == 12, "Expected 9 planned + 3 post-hoc in-sample configurations"
logger.info("In-sample rule configurations tried: %d (9 planned + 3 post hoc)", n_rule_trials)

extra_results.round(3)

INFO | Trial logged: in-sample, lookback=252, entry=1.5, exit=0.0, cost=5 bp → Sharpe 0.10
INFO | Trial logged: in-sample, lookback=252, entry=2.0, exit=0.0, cost=5 bp → Sharpe 0.07
INFO | Trial logged: in-sample, lookback=252, entry=2.5, exit=0.0, cost=5 bp → Sharpe 0.09
INFO | In-sample rule configurations tried: 12 (9 planned + 3 post hoc)


sharpe  ann_return  ann_vol  max_drawdown  n_trades  \
lookback entry_z                                                        
252      1.5       0.101       0.010    0.101        -0.488        35   
         2.0       0.066       0.006    0.093        -0.467        28   
         2.5       0.091       0.008    0.085        -0.413        22   

                  hit_rate  avg_holding_days  time_in_market  
lookback entry_z                                              
252      1.5         0.743           128.000           0.711  
         2.0         0.679           144.643           0.643  
         2.5         0.636           154.227           0.538

### 2.2.3: In-sample results
I tried entry at 1.5std, 2std and 2.5std with z-score lookbacks of 20, 60 and 120 days.
All nine have a positive Sharpe (0.05-0.50), 2std is the best threshold in every row, and longer lookbacks do better.
Performance peaks between 60 and 120 days and collapses at one year. This is consistent with the lack of evidence of long-run mean reversion, so a one-year window effectively bets
on a reversion that does not occur, while 60-120 day windows capture shorter-lived deviations
around a moving local mean. In total, 12 in-sample configurations were tried (9 planned, 3 post
hoc).

## 2.3: Out-of-sample test
I kept the baseline strategy keeping the hedge ratio frozen at its in-sample value (beta = 0.399) with 60-day rolling z-score, entry when |z| > 2 (the standard pairs-trading threshold), exit when z returns to 0.

In [19]:
prices_oos = prices.loc[pd.Timestamp(OOS_START):]
spread_full = compute_spread(prices, hedge, "KO", "PEP")
z_oos = rolling_zscore(spread_full, BASE_LOOKBACK).loc[pd.Timestamp(OOS_START):]
pos_oos = generate_positions(z_oos, BASE_ENTRY, BASE_EXIT)

assert hedge.fit_end <= IS_END, "Hedge ratio must come from the in-sample fit"
assert prices_oos.index.min() >= pd.Timestamp(OOS_START), "OOS frame must not contain in-sample dates"
assert pos_oos.index.equals(prices_oos.index)

oos_results = {}
for cost in COST_LEVELS:
    bt = backtest(prices_oos, hedge, "KO", "PEP", pos_oos, cost)
    metrics = performance_summary(bt, cost)
    log_trial(registry, "out-of-sample", BASE_LOOKBACK, BASE_ENTRY, BASE_EXIT, cost, "baseline OOS", hedge.beta, metrics)
    oos_results[f"{cost:g} bp"] = metrics

n_oos_trials = registry.execute("SELECT COUNT(*) FROM trials WHERE period = 'out-of-sample'").fetchone()[0]
assert n_oos_trials == len(COST_LEVELS), "Only the baseline, at three cost levels, may be run out-of-sample"

bt_oos = backtest(prices_oos, hedge, "KO", "PEP", pos_oos, BASE_COST)

comparison = pd.DataFrame(
    {
        f"in-sample 1990-2014, net of {BASE_COST:g} bp": baseline_results[f"{BASE_COST:g} bp"],
        f"out-of-sample 2015-2026, net of {BASE_COST:g} bp": oos_results[f"{BASE_COST:g} bp"],
    }
).T
comparison.style.format(
    {
        "sharpe": "{:.2f}", "ann_return": "{:.1%}", "ann_vol": "{:.1%}", "max_drawdown": "{:.1%}",
        "n_trades": "{:.0f}", "hit_rate": "{:.0%}", "avg_holding_days": "{:.0f}", "time_in_market": "{:.0%}",
    }
).set_caption(
    f"Baseline rule (60-day z-score, entry at 2std, exit at 0). The table is net of transaction costs of "
    f"{BASE_COST:g} bp per stock per trade"
)

INFO | Trial logged: out-of-sample, lookback=60, entry=2.0, exit=0.0, cost=0 bp → Sharpe -0.44
INFO | Trial logged: out-of-sample, lookback=60, entry=2.0, exit=0.0, cost=5 bp → Sharpe -0.50
INFO | Trial logged: out-of-sample, lookback=60, entry=2.0, exit=0.0, cost=10 bp → Sharpe -0.56


,sharpe,ann_return,ann_vol,max_drawdown,n_trades,hit_rate,avg_holding_days,time_in_market
"in-sample 1990-2014, net of 5 bp",0.45,4.2%,9.2%,-21.5%,106,75%,33,55%
"out-of-sample 2015-2026, net of 5 bp",-0.50,-2.9%,5.8%,-33.1%,40,55%,46,62%


In [20]:
def chained_equity(net_is: pd.Series, net_oos: pd.Series) -> pd.Series:
    """Chain in-sample and out-of-sample daily returns into one equity curve.

    Parameters
    ----------
    net_is : pd.Series
        Daily net returns, in-sample.
    net_oos : pd.Series
        Daily net returns, out-of-sample.

    Returns
    -------
    pd.Series
        Equity starting at 1 on the first in-sample day; the out-of-sample part
        starts from the last in-sample value.

    Notes
    -----
    The two backtests stay independent (the out-of-sample one starts flat and uses
    the frozen in-sample hedge ratio); chaining only affects how they are drawn.
    """
    assert net_is.index.max() < net_oos.index.min(), "In-sample must end before out-of-sample starts"
    return (1 + pd.concat([net_is, net_oos])).cumprod()


z_full = pd.concat([z_is, z_oos])

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=(
        f"z-score ({BASE_LOOKBACK}-day window) with entry thresholds ±{BASE_ENTRY:g}",
        "Equity curve, 1990-2026 (start = 1)",
    ),
)
fig.add_trace(go.Scatter(x=z_full.index, y=z_full, mode="lines", name="z-score", line_width=1), row=1, col=1)
for level in (-BASE_ENTRY, 0.0, BASE_ENTRY):
    fig.add_hline(y=level, line_dash="dash" if level else "solid", line_color="grey", line_width=1, row=1, col=1)

for cost in (0.0, BASE_COST):
    net_is = backtest(prices_is, hedge, "KO", "PEP", pos_is, cost)["net_return"]
    net_oos = backtest(prices_oos, hedge, "KO", "PEP", pos_oos, cost)["net_return"]
    equity = chained_equity(net_is, net_oos)
    fig.add_trace(go.Scatter(x=equity.index, y=equity, mode="lines", name=f"Equity, {cost:g} bp"), row=2, col=1)

fig.add_vline(x=pd.Timestamp(OOS_START), line_dash="dash", line_color="black", line_width=1)
fig.add_annotation(
    x=pd.Timestamp(OOS_START), y=1, yref="paper", text="out-of-sample →",
    showarrow=False, xanchor="left", yanchor="bottom",
)

fig.update_layout(
    title="Baseline rule: in-sample 1990-2014 and out-of-sample 2015-2026 (β frozen at in-sample value)",
    height=650,
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

## Phase 2: conclusions
The baseline does not survive out-of-sample. At 5 bp, the Sharpe falls from 0.45 (1990-2014) to −0.50 (2015-2026); annual return −2.9%, max drawdown −33.1%, 40 trades. It is negative before costs too (−0.44 at 0 bp), so the failure is not a cost effect. The hit rate drops from 75% to 55% and the average holding period rises from 33 to 46 days: divergences take longer to close, and often do not close. The equity curve is flat in 2015-2020 and declines steadily from 2021, faster in 2025-26.

This is consistent with the in-sample warnings, all documented before the out-of-sample was run: the spread is indistinguishable from a random walk, the half-life (~387 days) is far longer than the signal's window, the in-sample gain came mostly from one episode (1999-2004). The out-of-sample supports that the in-sample Sharpe was largely the product of one favourable episode, not a stable effect.
I did not test other grid configurations on this period. 

## 2.4: Trial registry

Every backtest that is run is written to a SQLite table, with the time of its first run. This makes the
search visible: anyone can count how many configurations were tried and check that all in-sample runs
happened before the single out-of-sample run.

In [21]:
trial_log = pd.read_sql_query(
    """
    SELECT period, lookback, entry_z, exit_z, cost_bps, label,
           ROUND(sharpe, 2) AS sharpe, n_trades, first_run_at
    FROM trials
    ORDER BY first_run_at, period, lookback, entry_z, cost_bps
    """,
    registry,
)

first_oos = trial_log.loc[trial_log["period"] == "out-of-sample", "first_run_at"].min()
last_is = trial_log.loc[trial_log["period"] == "in-sample", "first_run_at"].max()
assert last_is < first_oos, "Every in-sample configuration must have been run before the out-of-sample"
logger.info("Last new in-sample configuration: %s; first out-of-sample run: %s", last_is, first_oos)

trial_log

INFO | Last new in-sample configuration: 2026-10-10T12:30:58; first out-of-sample run: 2026-10-10T13:02:41


,period,lookback,entry_z,exit_z,cost_bps,label,sharpe,n_trades,first_run_at
0,in-sample,60,2.0,0.0,0.0,cost sensitivity,0.50,106,2026-10-10T12:07:43
1,in-sample,60,2.0,0.0,5.0,baseline,0.45,106,2026-10-10T12:07:43
2,in-sample,60,2.0,0.0,10.0,cost sensitivity,0.41,106,2026-10-10T12:07:43
3,in-sample,20,1.5,0.0,5.0,grid,0.20,329,2026-10-10T12:08:16
4,in-sample,20,2.0,0.0,5.0,grid,0.34,238,2026-10-10T12:08:16
5,in-sample,20,2.5,0.0,5.0,grid,0.05,125,2026-10-10T12:08:16
6,in-sample,60,1.5,0.0,5.0,grid,0.34,139,2026-10-10T12:08:16
7,in-sample,60,2.5,0.0,5.0,grid,0.20,61,2026-10-10T12:08:16
8,in-sample,120,1.5,0.0,5.0,grid,0.38,82,2026-10-10T12:08:16
9,in-sample,120,2.0,0.0,5.0,grid,0.50,66,2026-10-10T12:08:16


# Phase 3: Is the relationship stable?

**Method**: medium (rolling Engle-Granger), on the full 1990-2026 history: the out-of-sample has already been evaluated and nothing here changes the trading rule.

## 3.1: Unit-root check on each price
### How the Engle-Granger test works

Two prices can each wander like random walks and still be tied together: if some combination of them
keeps coming back to a fixed level, they are **cointegrated**. That is exactly the property a pairs
trade needs, because the spread is that combination.

The test has two steps.

**Step 1: estimate the relationship.** Regress KO on PEP by OLS:

$$KO_t = \alpha + \beta \, PEP_t + u_t$$

The residual $\hat u_t = KO_t - \hat\alpha - \hat\beta \, PEP_t$ is our spread (minus its mean).

**Step 2: test whether the residual is stationary.** Run an augmented Dickey-Fuller regression on it:

$$\Delta \hat u_t = \gamma \, \hat u_{t-1} + \sum_{i=1}^{p} \phi_i \, \Delta \hat u_{t-i} + \varepsilon_t$$

- $H_0: \gamma = 0$, the residual has a unit root: it can drift anywhere, so **no cointegration**.
- $H_1: \gamma < 0$, deviations are pulled back towards zero, so **cointegration**.

The lagged differences $\Delta \hat u_{t-i}$ absorb short-term autocorrelation; $p$ is chosen by an
information criterion.

**Why the critical values are different from a normal ADF test.** OLS picks $\beta$ to make the
residual as small and as stable as possible, so the residual looks more stationary than a random series
would. Using normal ADF critical values would find cointegration too often. Engle-Granger therefore uses
stricter (more negative) MacKinnon critical values: about -3.34 at 5% for two series with a constant,
against -2.86 for a plain ADF test. `statsmodels.coint` applies them automatically.

**Assumptions.**

1. Both prices are integrated of order one, I(1): non-stationary in levels, stationary in daily
   changes. This is why we first run ADF on levels and on changes.
2. The relationship is linear, with one $\alpha$ and one $\beta$ constant over the whole sample.
3. There is at most one cointegrating relationship (true by construction with two series).
4. The result can depend on which series is on the left side; we keep KO on PEP, as in Phase 2.

**Limits.** The test has low power: over short windows (like the 252-day rolling windows below) it can
fail to detect a relationship that exists. Not rejecting $H_0$ means "no evidence of cointegration",
not "proof that there is none".

In [22]:
from statsmodels.tsa.stattools import adfuller, coint
import warnings

warnings.filterwarnings("ignore", message="adfuller currently returns a plain tuple", category=FutureWarning)

TEST_MAXLAG = 5
PERIODS = (("in-sample 1990-2014", prices_is), ("out-of-sample 2015-2026", prices_oos))

unit_root = pd.DataFrame(
    {
        (period, series): {
            ticker: adfuller(
                frame[ticker].to_numpy() if series == "price level" else np.diff(frame[ticker].to_numpy()),
                maxlag=TEST_MAXLAG, regression="c", autolag=None,
            )[1]
            for ticker in TICKERS
        }
        for period, frame in PERIODS
        for series in ("price level", "daily change")
    }
)

levels = unit_root.xs("price level", axis=1, level=1)
changes = unit_root.xs("daily change", axis=1, level=1)
assert unit_root.stack().stack().between(0, 1).all(), "p-values must lie in [0, 1]"
assert (changes < 0.05).all().all(), "Daily price changes should be stationary"
logger.info("ADF p-values (levels should be > 0.05, daily changes < 0.05):\n%s", unit_root.round(4))

unit_root.style.format("{:.3f}").set_caption(
    "ADF p-values: price levels should not reject a unit root (p > 0.05), daily changes should (p < 0.05)"
)

INFO | ADF p-values (levels should be > 0.05, daily changes < 0.05):
    in-sample 1990-2014              out-of-sample 2015-2026             
            price level daily change             price level daily change
KO               0.9119          0.0                  0.9902          0.0
PEP              0.9939          0.0                  0.4876          0.0


Both conditions for the Engle-Granger test hold. Neither price level rejects the unit root (all p-values
well above 0.05), while daily price changes are clearly stationary (p-values close to zero).

## 3.2: Rolling regression and rolling Engle-Granger test
In each 252-day window I re-estimate the hedge ratio and test the residuals (the spread built with that window's β) for stationarity, using Engle-Granger.

In [23]:
def rolling_hedge_ratio(prices: pd.DataFrame, y: str, x: str, window: int) -> pd.Series:
    """Estimate the OLS hedge ratio of ``y`` on ``x`` over a trailing window.

    Parameters
    ----------
    prices : pd.DataFrame
        Price levels indexed by date.
    y : str
        Dependent stock.
    x : str
        Independent stock.
    window : int
        Window length in trading days.

    Returns
    -------
    pd.Series
        Rolling slope; NaN for the first ``window - 1`` days.

    Notes
    -----
    With an intercept, the OLS slope equals cov(y, x) / var(x), so it can be
    computed from rolling moments. Each value uses only the window ending at t.
    """
    cov = prices[y].rolling(window).cov(prices[x])
    var = prices[x].rolling(window).var()
    return (cov / var).rename("rolling_beta")


def rolling_coint_pvalue(
    prices: pd.DataFrame, y: str, x: str, window: int, step: int, maxlag: int
) -> pd.Series:
    """Run an Engle-Granger test on a trailing window, every ``step`` days.

    Parameters
    ----------
    prices : pd.DataFrame
        Price levels indexed by date.
    y : str
        Dependent stock.
    x : str
        Independent stock.
    window : int
        Window length in trading days.
    step : int
        Recompute every ``step`` days; values in between carry forward.
    maxlag : int
        Fixed number of lagged differences in the ADF regression on the residuals.

    Returns
    -------
    pd.Series
        Engle-Granger p-value (H0: no cointegration); NaN before the first full window.

    Notes
    -----
    ``coint`` regresses y on x in the window and runs an ADF test on the
    residuals, with Engle-Granger critical values: plain ADF ones would be too
    lenient, because the hedge ratio is fitted on the same data. Carrying values
    forward only fills the days between two computations, so no value uses data
    after its own date.
    """
    y_values = prices[y].to_numpy()
    x_values = prices[x].to_numpy()
    pvalues = pd.Series(np.nan, index=prices.index)
    for end in range(window, len(prices) + 1, step):
        sl = slice(end - window, end)
        pvalues.iloc[end - 1] = coint(y_values[sl], x_values[sl], trend="c", maxlag=maxlag, autolag=None)[1]
    if step > 1:
        pvalues = pvalues.ffill(limit=step - 1)
    return pvalues.rename("eg_pvalue")

Windows with an Engle-Granger p-value above 0.10 are flagged as "weak relationship"; the whole-in-sample test is reported for reference.

In [24]:
STAB_WINDOW = 252
TEST_STEP = 5
WEAK_PVALUE = 0.10

rolling_beta = rolling_hedge_ratio(prices, "KO", "PEP", STAB_WINDOW)
eg_p = rolling_coint_pvalue(prices, "KO", "PEP", STAB_WINDOW, TEST_STEP, TEST_MAXLAG)
weak = pd.Series(
    np.where(eg_p.isna(), np.nan, (eg_p > WEAK_PVALUE).astype(float)),
    index=eg_p.index,
    name="weak",
)

check_date = rolling_beta.dropna().index[-1]
window_data = prices.loc[:check_date].iloc[-STAB_WINDOW:]
beta_check = sm.OLS(window_data["KO"], sm.add_constant(window_data["PEP"])).fit().params["PEP"]
assert np.isclose(beta_check, rolling_beta.loc[check_date]), "Rolling β must equal the OLS slope on the same window"
assert eg_p.dropna().between(0, 1).all(), "p-values must lie in [0, 1]"
assert set(weak.dropna().unique()) <= {0.0, 1.0}

full_is_eg_p = coint(prices_is["KO"], prices_is["PEP"], trend="c", maxlag=TEST_MAXLAG, autolag=None)[1]
logger.info("Engle-Granger p-value on the whole in-sample: %.3f", full_is_eg_p)

stability_summary = pd.DataFrame(
    {
        "share of days flagged weak": [
            weak.loc[: pd.Timestamp(IS_END)].mean(),
            weak.loc[pd.Timestamp(OOS_START):].mean(),
        ],
        "median rolling β": [
            rolling_beta.loc[: pd.Timestamp(IS_END)].median(),
            rolling_beta.loc[pd.Timestamp(OOS_START):].median(),
        ],
    },
    index=["in-sample 1990-2014", "out-of-sample 2015-2026"],
)
stability_summary.style.format({"share of days flagged weak": "{:.0%}", "median rolling β": "{:.3f}"})

INFO | Engle-Granger p-value on the whole in-sample: 0.363


,share of days flagged weak,median rolling β
in-sample 1990-2014,88%,0.346
out-of-sample 2015-2026,89%,0.255


The Engle-Granger p-value on the whole in-sample is 0.363, does not find evidence of cointegration over
1990-2014, which confirms with a formal test what the crossing count found in Phase 2.

On rolling 252-day windows the relationship is flagged weak (p > 0.10) on 88% of in-sample days and
89% of out-of-sample days. 

The rolling hedge ratio drifts down over time: its median is 0.346 in-sample and 0.255
out-of-sample, against the frozen 0.399 used by the signal. The strategy in the out-of-sample therefore
traded about 1.5 times more PEP per KO share than the local relationship implied.
Recomputing the test every trading day instead of every 5 days gives the same shares of weak days (88% and 89%), so the choice of step does not affect the result.

## 3.3: Rolling hedge ratio and Engle-Granger p-value over time
Top: how far the window-by-window β drifts from the frozen β used by the signal. Bottom: when the relationship looks cointegrated (below the line) or weak (above).

In [25]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
    subplot_titles=(
        f"Rolling {STAB_WINDOW}-day hedge ratio (dashed: frozen in-sample β = {hedge.beta:.3f})",
        f"Rolling {STAB_WINDOW}-day Engle-Granger p-value (dashed: {WEAK_PVALUE:g} threshold)",
    ),
)
fig.add_trace(go.Scatter(x=rolling_beta.index, y=rolling_beta, mode="lines", name="Rolling β"), row=1, col=1)
fig.add_hline(y=hedge.beta, line_dash="dash", line_color="grey", row=1, col=1)
fig.add_trace(go.Scatter(x=eg_p.index, y=eg_p, mode="lines", name="Engle-Granger p-value", line_width=1), row=2, col=1)
fig.add_hline(y=WEAK_PVALUE, line_dash="dash", line_color="grey", row=2, col=1)
fig.add_vline(x=pd.Timestamp(OOS_START), line_dash="dash", line_color="black", line_width=1)

# Etichetta dentro il pannello alto, sotto il bordo: non collide più con il titolo del subplot
fig.add_annotation(
    x=pd.Timestamp(OOS_START), xref="x",
    y=0.98, yref="y domain",          # dominio dell'asse y del subplot 1, non 'paper'
    text="out-of-sample →", showarrow=False,
    xanchor="left", yanchor="top", xshift=4,
    font=dict(size=11, color="black"),
    bgcolor="rgba(255,255,255,0.8)",  #  anche se passa sopra la curva
)

fig.update_layout(
    title=dict(text="Stability of the KO/PEP relationship, 1990-2026", y=0.97),
    height=650, hovermode="x unified", template="plotly_white",
    margin=dict(t=90),
)
fig.show()

The rolling hedge ratio is very unstable as it swings between about -0.3 and
1.15, often within a few months, and turns negative in several years (1994-95, early 1997, 2006,
2017, 2025-26). The frozen 0.399 sits roughly in the
middle of these swings, but the local hedge ratio rarely stays near it. In the last year it falls to about -0.2, at the same time as the faster out-of-sample losses in 2025-26, to which the gap from the frozen 0.399 may have contributed.

The Engle-Granger p-value is high most of the time, and its dips below 0.10 are short and
scattered rather than forming recognisable regimes.
There is no visible break around 2015, which suggests that the relationship was already weak before the out-of-sample started.

## 3.4: Does the Phase 2 signal perform worse in weak periods?
Baseline returns are split by the flag known the day before each return, in-sample and out-of-sample separately; results are descriptive, since regimes last months.

In [26]:
def regime_performance(bt: pd.DataFrame, weak: pd.Series, periods_per_year: int = 252) -> pd.DataFrame:
    """Compare a backtest's returns in weak and strong relationship periods.

    Parameters
    ----------
    bt : pd.DataFrame
        Output of ``backtest``.
    weak : pd.Series
        1.0 where the relationship is flagged weak, 0.0 where not, NaN if unknown.
    periods_per_year : int, default 252
        Annualisation factor.

    Returns
    -------
    pd.DataFrame
        One row per regime: days, share of days, days in the market, annual
        return and Sharpe.

    Notes
    -----
    Day t's return is earned by the position decided at t-1, so it is matched
    with the flag known at t-1.
    """
    flag = weak.shift(1).reindex(bt.index)
    known_days = int(flag.notna().sum())
    rows = {}
    for name, value in (("strong (p ≤ 0.10)", 0.0), ("weak (p > 0.10)", 1.0)):
        mask = flag == value
        r = bt.loc[mask, "net_return"]
        vol = r.std() * np.sqrt(periods_per_year)
        rows[name] = {
            "days": int(mask.sum()),
            "share_of_days": mask.sum() / known_days if known_days else float("nan"),
            "days_in_market": int((bt.loc[mask, "held"] != 0).sum()),
            "ann_return": r.mean() * periods_per_year,
            "sharpe": r.mean() * periods_per_year / vol if vol > 0 else float("nan"),
        }
    return pd.DataFrame(rows).T


regimes = pd.concat(
    {
        "in-sample 1990-2014": regime_performance(bt_is, weak),
        "out-of-sample 2015-2026": regime_performance(bt_oos, weak),
    }
)
regimes.style.format(
    {"days": "{:.0f}", "share_of_days": "{:.0%}", "days_in_market": "{:.0f}", "ann_return": "{:.1%}", "sharpe": "{:.2f}"}
)

This table shows that trading on periods flagged as strong would have lost money in the in-sample-data, making it a useless trading filter.

## Phase 3: conclusion
The relationship between KO and PEP does not look stable. On the whole in-sample the Engle-Granger
test finds no evidence of cointegration (p = 0.363), one-year windows reject no cointegration on only
11-12% of days, and the rolling hedge ratio swings between about -0.3 and 1.15. The
Phase 2 signal does not perform consistently worse in the flagged weak periods: the comparison
goes in opposite directions in-sample and out-of-sample. The analysis finds no convincing evidence of a stable
relationship, but it does not identify a single cause of the out-of-sample loss; the gap between the
frozen hedge ratio and the rolling one may have contributed, but the rolling hedge ratio is a
diagnostic, not proof of the cause.

# Phase 4: Institutional positioning, short interest
The brief's free source (Nasdaq) covers only Nasdaq-listed stocks, so it has PEP but not KO (NYSE). I found out that FINRA publishes short interest for all exchange-listed stocks, so I use it for both stocks and use the Nasdaq data to cross-check PEP.

## 4.1: Nasdaq short interest for PEP

In [27]:
import json
import urllib.error
import urllib.request

NASDAQ_SI_URL = "https://api.nasdaq.com/api/quote/PEP/short-interest?assetClass=stocks"
NASDAQ_SI_PATH = Path("data/raw/pep_short_interest.json")


def fetch_nasdaq_short_interest(url: str, path: Path, refresh: bool = False) -> dict:
    """Download Nasdaq short-interest JSON, or load it from the checkpoint.

    Parameters
    ----------
    url : str
        Nasdaq short-interest endpoint for one ticker.
    path : Path
        Location of the raw JSON checkpoint.
    refresh : bool, default False
        If True, ignore the checkpoint and download again.

    Returns
    -------
    dict
        Raw JSON payload.

    Raises
    ------
    DataFetchError
        If the request fails or the response is not valid JSON.

    Notes
    -----
    The source keeps only a rolling ~12-month window, so the first download is
    saved and reused. A browser-like User-Agent is sent because the endpoint
    rejects default clients.
    """
    if path.exists() and not refresh:
        logger.info("Loading Nasdaq short-interest checkpoint %s", path)
        return json.loads(path.read_text(encoding="utf-8"))

    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except (urllib.error.URLError, json.JSONDecodeError) as exc:
        raise DataFetchError(f"Nasdaq short-interest download failed: {exc}") from exc

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    logger.info("Saved Nasdaq short-interest checkpoint %s", path)
    return payload


def parse_nasdaq_short_interest(payload: dict, as_of: date) -> pd.Series:
    """Extract short interest by settlement date from the Nasdaq payload.

    Parameters
    ----------
    payload : dict
        Output of ``fetch_nasdaq_short_interest``.
    as_of : date
        No settlement date after this is kept.

    Returns
    -------
    pd.Series
        Short interest in shares, indexed by settlement date.

    Raises
    ------
    DataFetchError
        If the payload does not have the expected structure or has no rows.
    """
    try:
        rows = payload["data"]["shortInterestTable"]["rows"]
    except (KeyError, TypeError) as exc:
        raise DataFetchError(f"Unexpected Nasdaq payload; status: {payload.get('status')}") from exc
    if not rows:
        raise DataFetchError("The Nasdaq short-interest table is empty")

    raw = pd.DataFrame(rows)
    values = pd.to_numeric(raw["interest"].astype(str).str.replace(",", "", regex=False), errors="coerce")
    series = pd.Series(values.to_numpy(), index=pd.to_datetime(raw["settlementDate"], format="%m/%d/%Y"), name="PEP")
    series = series.sort_index().loc[lambda s: s.index <= pd.Timestamp(as_of)]

    assert series.index.is_unique, "Settlement dates must be unique"
    assert series.notna().all() and (series > 0).all(), "Short interest must be positive"
    return series


nasdaq_pep_si = parse_nasdaq_short_interest(fetch_nasdaq_short_interest(NASDAQ_SI_URL, NASDAQ_SI_PATH), AS_OF)
logger.info(
    "Nasdaq PEP short interest: %d dates from %s to %s",
    len(nasdaq_pep_si), nasdaq_pep_si.index.min().date(), nasdaq_pep_si.index.max().date(),
)

INFO | Loading Nasdaq short-interest checkpoint data\raw\pep_short_interest.json
INFO | Nasdaq PEP short interest: 24 dates from 2025-10-15 to 2026-09-30


The nasdaq dataset only lists the last 24 dates for PEP. 

## 4.2: FINRA short interest for KO and PEP

In [28]:
FINRA_SI_PATH = Path("data/raw/finra_short_interest.csv")


def load_finra_short_interest(path: Path, symbols: tuple[str, ...], as_of: date) -> pd.DataFrame:
    """Load a FINRA short-interest export for the requested symbols.

    Parameters
    ----------
    path : Path
        CSV exported from FINRA's Equity Short Interest page.
    symbols : tuple[str, ...]
        Tickers to keep.
    as_of : date
        No settlement date after this is kept.

    Returns
    -------
    pd.DataFrame
        Indexed by settlement date, with MultiIndex columns (field, symbol) for
        short_interest, avg_daily_volume and days_to_cover.

    Raises
    ------
    DataFetchError
        If expected columns are missing, a symbol has no rows, or a
        (date, symbol) pair appears more than once.

    Notes
    -----
    Rows flagged "R" are revisions. In this export each (date, symbol) appears
    once, so revised rows already carry the corrected value; the duplicate check
    guards against an export that contains both versions.
    """
    raw = pd.read_csv(path)
    needed = {"Settlement Date", "Symbol", "Current Short", "Avg Daily Vol", "Days to Cover"}
    if not needed <= set(raw.columns):
        raise DataFetchError(f"Missing columns {needed - set(raw.columns)}; found: {list(raw.columns)}")

    raw = raw[raw["Symbol"].isin(symbols)].copy()
    raw["Settlement Date"] = pd.to_datetime(raw["Settlement Date"])
    if raw.duplicated(["Settlement Date", "Symbol"]).any():
        raise DataFetchError("Duplicate (date, symbol) rows: the export contains revised and original values")

    wide = raw.pivot(
        index="Settlement Date",
        columns="Symbol",
        values=["Current Short", "Avg Daily Vol", "Days to Cover"],
    ).rename(columns={"Current Short": "short_interest", "Avg Daily Vol": "avg_daily_volume", "Days to Cover": "days_to_cover"})
    wide = wide.sort_index().loc[lambda df: df.index <= pd.Timestamp(as_of)]

    missing = [s for s in symbols if s not in wide["short_interest"].columns]
    if missing:
        raise DataFetchError(f"No FINRA rows for {missing}")
    assert wide["short_interest"].notna().all().all(), "Both symbols must have a value on every settlement date"
    assert (wide["short_interest"] > 0).all().all(), "Short interest must be positive"
    return wide


finra = load_finra_short_interest(FINRA_SI_PATH, TICKERS, AS_OF)
n_revised = int(pd.read_csv(FINRA_SI_PATH)["Revision Flag"].eq("R").sum())
logger.info(
    "FINRA short interest: %d settlement dates from %s to %s (%d revised rows)",
    len(finra), finra.index.min().date(), finra.index.max().date(), n_revised,
)

common = finra.index.intersection(nasdaq_pep_si.index)
gap = (finra.loc[common, ("short_interest", "PEP")] - nasdaq_pep_si.loc[common]).abs() / nasdaq_pep_si.loc[common]
assert len(common) >= 20, "Most Nasdaq dates should appear in the FINRA file"
assert gap.max() < 0.01, "FINRA and Nasdaq should report the same PEP short interest"
logger.info("PEP cross-check on %d common dates: max relative difference %.4f%%", len(common), 100 * gap.max())

finra["days_to_cover"].describe().style.format("{:.2f}")

INFO | FINRA short interest: 211 settlement dates from 2017-12-29 to 2026-09-30 (15 revised rows)
INFO | PEP cross-check on 24 common dates: max relative difference 0.0000%


Symbol,KO,PEP
count,211.00,211.00
mean,2.21,2.48
std,0.63,0.67
min,1.00,1.01
25%,1.75,1.97
50%,2.13,2.44
75%,2.66,2.94
max,4.26,4.45


The FINRA file has 211 settlement dates for each stock, from December 2017 to September 2026
(15 rows are revisions, already carrying the corrected value). PEP matches the Nasdaq source exactly
on all 24 common dates, so the two sources report the same data and the FINRA values for KO can be
trusted in the same way.

Levels are not comparable across the two stocks, but changes in their ratio are so I compare the change in log(SI PEP / SI KO) between settlement dates with KO's log return minus PEP's over the same interval. We want to check if a relative change in short interest (PEP vs KO) moves together with a relative change in prices (KO vs PEP) over the interval the short interest is measured

In [29]:
from IPython.display import display

si = finra["short_interest"]
rel_si = (np.log(si["PEP"]) - np.log(si["KO"])).rename("log_rel_si")

px_settle = prices.reindex(si.index, method="ffill")
assert px_settle.notna().all().all(), "Every settlement date must have a price"
rel_px = (np.log(px_settle["KO"]) - np.log(px_settle["PEP"])).rename("log_rel_px")

si_changes = pd.DataFrame(
    {
        "d_rel_si": rel_si.diff(),
        "d_rel_px": rel_px.diff(),
        "zscore": rolling_zscore(spread_full, BASE_LOOKBACK).reindex(si.index, method="ffill"),
        "weak": weak.reindex(si.index, method="ffill"),
    }
).dropna(subset=["d_rel_si", "d_rel_px"])

si_corr = si_changes["d_rel_si"].corr(si_changes["d_rel_px"])
si_threshold = 2 / np.sqrt(len(si_changes))
logger.info(
    "Correlation of relative short-interest changes with relative price changes: %+.3f "
    "(n = %d, rough 5%% threshold ±%.3f)",
    si_corr, len(si_changes), si_threshold,
)

largest_moves = si_changes.reindex(si_changes["d_rel_si"].abs().sort_values(ascending=False).index).head(8)
largest_view = largest_moves.rename(
    columns={
        "d_rel_si": "change in relative short interest",
        "d_rel_px": "change in relative price",
        "zscore": "Phase 2 spread z-score",
        "weak": "Phase 3 weak flag",
    }
)
largest_view.index = largest_view.index.date
largest_view.style.format(
    {
        "change in relative short interest": "{:+.3f}",
        "change in relative price": "{:+.3f}",
        "Phase 2 spread z-score": "{:+.2f}",
        "Phase 3 weak flag": "{:.0f}",
    }
).set_caption("Eight largest changes in relative short interest")

INFO | Correlation of relative short-interest changes with relative price changes: -0.066 (n = 210, rough 5% threshold ±0.138)


,change in relative short interest,change in relative price,Phase 2 spread z-score,Phase 3 weak flag
2021-06-15,+0.418,-0.002,-0.48,1
2026-02-27,-0.406,+0.013,+1.05,1
2018-07-31,+0.394,+0.021,-0.45,1
2026-09-30,-0.374,+0.037,+1.24,1
2021-03-31,-0.345,-0.029,+0.37,1
2018-06-15,+0.336,-0.036,-0.64,1
2025-02-14,-0.312,+0.131,+3.04,1
2019-04-30,-0.310,+0.008,-0.50,1


## 4.4: Chart

In [30]:
z_window = rolling_zscore(spread_full, BASE_LOOKBACK).loc[si.index.min():]

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    subplot_titles=(
        "Short interest (millions of shares, FINRA)",
        "Relative short interest log(PEP/KO) and relative price log(KO/PEP), change since Dec 2017",
        f"Spread z-score ({BASE_LOOKBACK}-day window), dashed: ±{BASE_ENTRY:g}",
    ),
)
for ticker in TICKERS:
    fig.add_trace(go.Scatter(x=si.index, y=si[ticker] / 1e6, mode="lines", name=f"{ticker} short interest"), row=1, col=1)
fig.add_trace(go.Scatter(x=rel_si.index, y=rel_si - rel_si.iloc[0], mode="lines",
                         name="Shorting of PEP relative to KO"), row=2, col=1)
fig.add_trace(go.Scatter(x=rel_px.index, y=rel_px - rel_px.iloc[0], mode="lines",
                         name="KO outperformance over PEP"), row=2, col=1)
fig.add_trace(go.Scatter(x=z_window.index, y=z_window, mode="lines", name="Spread z-score", line_width=1), row=3, col=1)
for level in (-BASE_ENTRY, BASE_ENTRY):
    fig.add_hline(y=level, line_dash="dash", line_color="grey", line_width=1, row=3, col=1)
fig.update_layout(
    title="Phase 4: short interest of KO and PEP against the pair, 2017-2026",
    height=800, hovermode="x unified", template="plotly_white",
)
fig.show()

## Phase 4: conclusions

Over the 210 intervals the correlation between the change in relative short interest (PEP vs KO) and the change in relative price (KO vs PEP) is -0.066. That is well inside the ±0.14 range you would expect from chance alone.

So in this data, changes in short interest do not tell us anything about how the pair moves. This is a same-period comparison, not a test of whether short interest predicts prices, and both stocks are only lightly shorted.


# Phase 5: Simulating the pair

Method: joint block bootstrap. I resample blocks of daily log returns of KO and PEP from the in-sample period, taking the same blocks for both stocks so their correlation and short-term behaviour are preserved, rebuild synthetic prices, and rerun the full Phase 2 procedure on each path (estimate the hedge ratio, build the spread, apply the baseline rule with 5 bp costs). I chose this over an Ornstein-Uhlenbeck simulation because an OU spread mean-reverts by construction, so the signal would work on it almost by definition.

Shuffling blocks of 10 days keeps everything that happens within two weeks but destroys any slower structure, including mean reversion over the few weeks the signal trades on. The simulated paths are therefore a benchmark in which the two stocks co-move day to day but the spread has no mean reversion beyond the block length. I try 1000 paths at 10 days block length and then check if anything changes with 5 or 20 days blocks. The simulation uses in-sample returns only, because it tests whether the in-sample result could be luck

## 5.1: Bootstrap functions

In [31]:
def joint_block_bootstrap(log_returns: pd.DataFrame, block: int, rng: np.random.Generator) -> np.ndarray:
    """Resample whole blocks of consecutive days, the same blocks for every column.

    Parameters
    ----------
    log_returns : pd.DataFrame
        Daily log returns, one column per stock.
    block : int
        Block length in trading days.
    rng : np.random.Generator
        Random number generator.

    Returns
    -------
    np.ndarray
        Resampled returns with the same shape as ``log_returns``.

    Notes
    -----
    Taking the same block start for both stocks keeps their correlation and each
    series' short-term patterns (such as volatility clustering) within a block.
    Blocks are drawn with replacement from all possible start dates and
    concatenated, then cut to the original length.
    """
    values = log_returns.to_numpy()
    n = len(values)
    n_blocks = int(np.ceil(n / block))
    starts = rng.integers(0, n - block + 1, size=n_blocks)
    index = (starts[:, None] + np.arange(block)).ravel()[:n]
    return values[index]


def simulate_baseline_sharpe(
    real_prices: pd.DataFrame,
    log_returns: pd.DataFrame,
    block: int,
    n_paths: int,
    seed: int,
) -> pd.DataFrame:
    """Run the Phase 2 baseline on joint block-bootstrap paths of the two prices.

    Parameters
    ----------
    real_prices : pd.DataFrame
        Real prices; their first row is the starting point of every path and
        their index gives the dates.
    log_returns : pd.DataFrame
        Daily log returns to resample (one row fewer than ``real_prices``).
    block : int
        Block length in trading days.
    n_paths : int
        Number of simulated paths.
    seed : int
        Random seed, for reproducibility.

    Returns
    -------
    pd.DataFrame
        One row per path: beta, sharpe, ann_return, n_trades. Paths with a
        non-positive hedge ratio have NaN metrics.

    Notes
    -----
    Each path repeats the full real procedure: OLS hedge ratio on the path,
    spread, 60-day z-score, entry at 2, exit at 0, 5 bp costs. The hedge ratio is
    estimated directly, without the positivity assert of ``fit_hedge_ratio``:
    a path with beta <= 0 is recorded and excluded, because on real data that
    case would have stopped the analysis rather than produced a trade.
    """
    assert len(log_returns) == len(real_prices) - 1, "Returns must have one row fewer than prices"
    rng = np.random.default_rng(seed)
    start = real_prices.iloc[0].to_numpy()
    rows = []
    for _ in range(n_paths):
        sampled = joint_block_bootstrap(log_returns, block, rng)
        levels = start * np.exp(np.vstack([np.zeros(2), np.cumsum(sampled, axis=0)]))
        path = pd.DataFrame(levels, index=real_prices.index, columns=log_returns.columns)

        slope, intercept = np.polyfit(path["PEP"], path["KO"], 1)
        if slope <= 0:
            rows.append({"beta": slope, "sharpe": np.nan, "ann_return": np.nan, "n_trades": np.nan})
            continue
        fit = HedgeFit(
            alpha=float(intercept), beta=float(slope), n_obs=len(path),
            fit_start=path.index.min().date(), fit_end=path.index.max().date(),
        )
        z = rolling_zscore(compute_spread(path, fit, "KO", "PEP"), BASE_LOOKBACK)
        positions = generate_positions(z, BASE_ENTRY, BASE_EXIT)
        metrics = performance_summary(backtest(path, fit, "KO", "PEP", positions, BASE_COST), BASE_COST)
        rows.append({"beta": fit.beta, "sharpe": metrics["sharpe"], "ann_return": metrics["ann_return"], "n_trades": metrics["n_trades"]})
    return pd.DataFrame(rows)

## 5.2: Running the simulation

In [32]:
BOOT_PATHS = 1000
BOOT_BLOCKS = (5, 10, 20)
BOOT_SEED = 42

is_log_returns = np.log(prices_is[list(TICKERS)]).diff().dropna()
assert is_log_returns.index.max() <= pd.Timestamp(IS_END), "The bootstrap must use in-sample returns only"

real_is_sharpe = baseline_results[f"{BASE_COST:g} bp"]["sharpe"]
simulations = {
    block: simulate_baseline_sharpe(prices_is[list(TICKERS)], is_log_returns, block, BOOT_PATHS, BOOT_SEED)
    for block in BOOT_BLOCKS
}

rows = {}
for block, sims in simulations.items():
    valid = sims["sharpe"].dropna()
    rows[f"{block}-day blocks"] = {
        "valid paths": len(valid),
        "paths with beta <= 0": int(sims["sharpe"].isna().sum()),
        "median Sharpe": valid.median(),
        "5th percentile": valid.quantile(0.05),
        "95th percentile": valid.quantile(0.95),
        "share of paths with Sharpe > 0": (valid > 0).mean(),
        "share of paths beating the real Sharpe": (valid >= real_is_sharpe).mean(),
    }
    assert len(valid) > 0.9 * BOOT_PATHS, f"Too many invalid paths with {block}-day blocks"

logger.info("Real in-sample Sharpe of the baseline (5 bp): %.2f", real_is_sharpe)
bootstrap_summary = pd.DataFrame(rows).T
bootstrap_summary.style.format(
    {
        "valid paths": "{:.0f}", "paths with beta <= 0": "{:.0f}", "median Sharpe": "{:.2f}",
        "5th percentile": "{:.2f}", "95th percentile": "{:.2f}",
        "share of paths with Sharpe > 0": "{:.0%}", "share of paths beating the real Sharpe": "{:.0%}",
    }
)

INFO | Real in-sample Sharpe of the baseline (5 bp): 0.45


,valid paths,paths with beta <= 0,median Sharpe,5th percentile,95th percentile,share of paths with Sharpe > 0,share of paths beating the real Sharpe
5-day blocks,989,11,0.08,-0.27,0.38,64%,2%
10-day blocks,994,6,0.15,-0.15,0.47,77%,6%
20-day blocks,996,4,0.23,-0.09,0.54,89%,12%


## 5.3: Distribution of simulated Sharpe ratios

In [33]:
fig = go.Figure()
for block, sims in simulations.items():
    fig.add_trace(go.Histogram(x=sims["sharpe"].dropna(), name=f"{block}-day blocks", opacity=0.55, nbinsx=40))
fig.add_vline(
    x=real_is_sharpe, line_dash="dash", line_color="black",
    annotation_text=f"real in-sample Sharpe = {real_is_sharpe:.2f}", annotation_position="top right",
)
fig.update_layout(
    barmode="overlay",
    title=f"Baseline Sharpe on {BOOT_PATHS} joint block-bootstrap paths per block length (in-sample returns, 5 bp)",
    xaxis_title="Annualised Sharpe ratio",
    yaxis_title="Number of paths",
    template="plotly_white",
)
fig.show()

## Phase 5: conclusions
Because the simulated histories are built from shuffled 10-day blocks, any tendency of the spread to return to its mean over more than about two weeks is removed by construction, while patterns within a block are kept. The trades last 33 days on average, so most of the mean reversion the strategy relies on is not present in the simulations. Yet the median Sharpe is positive (0.15 with 10-day blocks) and most histories end in profit. The likely reason is that the hedge ratio is estimated on the same period the strategy trades on, which makes the spread look more stable than it really was. So the real Sharpe of 0.45 should be compared with the simulations, not with zero. This means that the strategy makes money even without mean reversion. 

I ran the strategy on 1000 simulated histories, each without any real edge by construction, and compared their Sharpe ratios with the 0.45 from the real history. With 10-day blocks, about 6% of the simulations beat 0.45, so the real result would rank around 60th which is near the top, but not first. A genuine edge would place it above almost all simulations (below 1%). 
The result is borderline thus we can’t conclude it’s luck, but we can’t rule it out either.

Most of the money the strategy made in-sample could be explained by two things that have nothing to do with a special link between KO and PEP: the hedge ratio was calculated on the same years the strategy traded on, and two related stocks naturally move together in the short term. On top of that, Phase 2 showed that most of the profit came from a single period, 1999-2004. Once these are taken into account, what is left is too small to say that there is a real relationship between KO and PEP that can be traded. This fits with the strategy losing money out-of-sample.